# Rosely 10Eros-Max beta_5 / MiniMax H3 Ref2VA — Vast Serverless Test

This notebook calls the deployed Vast.ai Serverless `/generate/sync` endpoint.

The worker uses **10Eros-Max beta_5 non-Turbo INT8** with the Blackwell NVFP4-AWQ Qwen3-VL encoder.

For the first smoke test use **480×864, 5 seconds, 8 steps, simple scheduler**.

In [ ]:
%pip install -q "vastai[serverless]" requests pillow

In [ ]:
import os
import json
import uuid
import base64
from pathlib import Path
from getpass import getpass

if not os.getenv("VAST_API_KEY"):
    os.environ["VAST_API_KEY"] = getpass("VAST_API_KEY: ")

ENDPOINT_NAME = "rosely-minimax-h3-ref2va"
print("Endpoint:", ENDPOINT_NAME)

## Choose an input image

Use either a URL accessible by the worker or a local file. If both are set, the URL is used.

In [ ]:
INPUT_IMAGE_URL = ""
LOCAL_IMAGE_PATH = ""

def image_input():
    if INPUT_IMAGE_URL.strip():
        return {"input_image_url": INPUT_IMAGE_URL.strip()}
    if LOCAL_IMAGE_PATH.strip():
        raw = Path(LOCAL_IMAGE_PATH).read_bytes()
        return {"input_image_base64": base64.b64encode(raw).decode("ascii")}
    raise ValueError("Set INPUT_IMAGE_URL or LOCAL_IMAGE_PATH")

## Build the request

The server automatically adds `<Picture 1>` if you omit it. The old HMNSFW LoRA fields are no longer required.

In [ ]:
payload = {
    "input": {
        "request_id": f"h3_test_{uuid.uuid4().hex[:12]}",
        **image_input(),
        "prompt": (
            "Preserve the subject identity, facial features, body proportions, "
            "lighting and visual style from the reference. "
            "Natural coherent motion, stable anatomy, smooth temporal consistency."
        ),
        "width": 480,
        "height": 864,
        "duration_seconds": 5,
        "steps": 8,
        "scheduler": "simple",
        "ref_image_size": "match",
        "include_audio": True,
        # "seed": 123456789,
    }
}

safe_payload = {**payload, "input": dict(payload["input"])}
if "input_image_base64" in safe_payload["input"]:
    safe_payload["input"]["input_image_base64"] = "<omitted>"
print(json.dumps(safe_payload, indent=2))

In [ ]:
from vastai import Serverless

async def run_generation():
    async with Serverless(
        api_key=os.environ["VAST_API_KEY"],
        default_request_timeout=4200,
    ) as client:
        endpoint = await client.get_endpoint(name=ENDPOINT_NAME)
        return await endpoint.request(
            "/generate/sync",
            payload,
            cost=100,
            retry=True,
        )

result = await run_generation()
print(json.dumps(result, indent=2))

In [ ]:
if not result.get("ok"):
    raise RuntimeError(f"Vast request failed: status={result.get('status')} text={result.get('text')}")

body = result["response"]
print("Model      :", body.get("model"))
print("Request ID :", body["request_id"])
print("Generation :", body["generation_seconds"], "seconds")
print("Seed       :", body["seed"])
print("S3 URI     :", body["s3_uri"])
print("Expires in :", body["output_url_expires_in_seconds"], "seconds")
print("\nPresigned URL:\n", body["output_url"])

In [ ]:
import requests
from IPython.display import Video, display

video_response = requests.get(body["output_url"], timeout=300)
video_response.raise_for_status()
output_file = Path(f'{body["request_id"]}.mp4')
output_file.write_bytes(video_response.content)
print("Saved:", output_file, f"({output_file.stat().st_size / 1024 / 1024:.1f} MiB)")
display(Video(str(output_file), embed=True))

## Higher-quality test

After the smoke test succeeds, try:

```python
payload["input"].update({
    "width": 768,
    "height": 1344,
    "duration_seconds": 5,
    "steps": 8,
    "scheduler": "simple",
    "ref_image_size": "max",
})
```

`ref_image_size="max"` gives the reference path more image detail but uses more memory.